# adaptive-intelligence — Harness Agent & Loop Engineering Demo

This notebook shows how the system evaluates **every pipeline decision**, not just the final answer.

**Two modes:**
- Part A: Without LLM — see harness evaluating retrieval decisions (fast, no GPU)
- Part B: With HuggingFace — see harness + synthesized answers (needs GPU)

**Before running:** Runtime > Change runtime type > T4 GPU

---
[PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | @VK_Venkatkumar


## 1. Fix Colab (run once, restart runtime, then skip)

In [1]:
!pip uninstall torchvision torchaudio -y -q
!pip install torchvision torchaudio -q
print('Now: Runtime > Restart runtime, then run from cell 2')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5

## 2. Install

In [1]:
%%capture
!pip install adaptive-intelligence chromadb transformers accelerate -q


## 3. Create Documents

In [2]:
import os

os.makedirs('/content/docs', exist_ok=True)

docs = {
    "q3_report.txt": "NovaTech Q3 2025: Revenue $847M (+12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Operating income $127M, margin 15.0%. EBITDA $168M, margin 19.8%. Cash $1.2B. Q4 guidance: $870-890M.",
    "risks.txt": "Risk Assessment 2025: Supply Chain (HIGH) - 65% dependency on Meridian Semiconductors. Pacific Chip Alliance secondary supplier, target 45% by Q2 2026. Cybersecurity (MEDIUM) - CyberShield Partners, $12M zero-trust. Market (MEDIUM) - AscentTech competition, share declined 28% to 26%.",
    "medical.txt": "Treatment Protocol: Type 2 Diabetes. First-line: Metformin 500mg twice daily. If HbA1c above 7.5% after 3 months, add Empagliflozin 10mg. Monitor eGFR every 6 months. Contraindicated if eGFR below 30. Drug interaction: Metformin + contrast dye = lactic acidosis risk, hold 48 hours.",
}

for name, content in docs.items():
    with open(f'/content/docs/{name}', 'w') as f:
        f.write(content)

print(f'{len(docs)} documents created')


3 documents created


---
# PART A: Without LLM — Harness Evaluating Retrieval Decisions

Fast. No GPU needed. Shows how harness scores every decision.


## 4. Setup Engine (no LLM)

In [3]:
from adaptive_intelligence import AdaptiveAI
import logging
logging.getLogger('adaptive_intelligence').setLevel(logging.ERROR)

engine = AdaptiveAI(
    llm_backend='none',
    vectorless=True,
    storage_dir='/content/state_nollm',
    log_level='ERROR',
)

engine.ingest('/content/docs')
print(f'Ready. Graph: {engine.graph.node_count} nodes')


Ready. Graph: 23 nodes


## 5. Harness in Action — Every Decision Evaluated

Each query shows ✓ decisions that helped and ✗ decisions that were wasted.


In [4]:
queries = [
    ('What is Q3 revenue?', 'factual'),
    ('How is Meridian connected to supply chain risk?', 'relational'),
    ('Compare revenue vs EBITDA margin', 'comparative'),
    ('What is Metformin dosage?', 'medical'),
    ('What drug interactions with Metformin?', 'medical-relational'),
]

for query, qtype in queries:
    r = engine.ask(query)
    print(f'Q: {query}  [{qtype}]')
    print(f'  Strategy: {r.retrieval_strategy}')
    print(f'  Confidence: {r.confidence:.0%}')
    print(f'  Answer: {r.answer[:120]}...')

    if hasattr(r, 'harness_report') and r.harness_report:
        report = r.harness_report
        print(f'  Harness efficiency: {report.efficiency:.0%}')
        for d in report.decisions:
            mark = '✓' if d.helped else '✗'
            print(f'    {mark} {d.decision}: {d.name} ({d.impact:+.2f}) {d.detail}')
        if report.recommendations:
            for rec in report.recommendations[:2]:
                print(f'    → {rec}')

    if hasattr(r, 'shaped_reward'):
        print(f'  Shaped reward: {r.shaped_reward:.2f} (raw: {r.confidence:.2f})')
    print()


Q: What is Q3 revenue?  [factual]
  Strategy: table_first + depth=1
  Confidence: 78%
  Answer: From q3_report.txt:
NovaTech Q3 2025: Revenue $847M (+12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Operat...
  Harness efficiency: 100%
    ✓ route: table_first (+0.23) route matches query type
    ✓ depth: 5 (+0.10) good utilization (100%)
    ✓ graph: off (+0.05) graph skipped (saved compute)
  Shaped reward: 0.65 (raw: 0.78)

Q: How is Meridian connected to supply chain risk?  [relational]
  Strategy: page_graph + graph(2-hop) + depth=2
  Confidence: 70%
  Answer: From risks.txt:
Risk Assessment 2025: Supply Chain (HIGH) - 65% dependency on Meridian Semiconductors. Pacific Chip Alli...
  Harness efficiency: 67%
    ✓ route: page_graph (+0.21) route matches query type
    ✓ depth: 5 (+0.10) good utilization (100%)
    ✗ graph: on (-0.10) graph activated but returned no context
    → Graph decision: graph activated but returned no context
  Shaped reward: 0.56 (raw: 0.70)

Q:

## 6. Loop Engineering — Exploration Adapts Per Domain


In [5]:
# Run more queries
for q in ['EBITDA?', 'Revenue?', 'Margin?', 'Cash?', 'Q4 guidance?',
          'Product revenue?', 'Services?', 'Net income?',
          'Empagliflozin dosage?', 'eGFR monitoring?']:
    engine.ask(q)

stats = engine._loop_engineer.get_stats()
print(f'Total queries: {stats["global_queries"]}')
print(f'Domains tracked: {stats["domain_count"]}\n')

for domain, data in stats['domains'].items():
    print(f'{domain}:')
    print(f'  Queries: {data["queries"]}, Exploration: {data["exploration"]}, Warmup left: {data["warmup_left"]}')


Total queries: 15
Domains tracked: 6

financial:structured_lookup:
  Queries: 4, Exploration: 100.0%, Warmup left: 11
operational:relational:
  Queries: 1, Exploration: 100.0%, Warmup left: 14
financial:comparative:
  Queries: 1, Exploration: 100.0%, Warmup left: 14
general:factual:
  Queries: 7, Exploration: 100.0%, Warmup left: 8
healthcare:factual:
  Queries: 1, Exploration: 100.0%, Warmup left: 14
financial:factual:
  Queries: 1, Exploration: 100.0%, Warmup left: 14


## 7. Harness Stats — What Helped, What Was Wasted


In [6]:
h = engine._harness.get_stats()
print(f'Reports: {h["reports"]}')
if 'avg_efficiency' in h:
    print(f'Avg efficiency: {h["avg_efficiency"]}')

if 'decision_stats' in h:
    print(f'\n{"Decision":<30} {"Count":<8} {"Help Rate":<12} {"Avg Impact"}')
    print('-' * 65)
    for k, v in h['decision_stats'].items():
        print(f'{k:<30} {v["count"]:<8} {v["help_rate"]:<12} {v["avg_impact"]}')


Reports: 15
Avg efficiency: 98%

Decision                       Count    Help Rate    Avg Impact
-----------------------------------------------------------------
graph:off                      14       100%         0.05
depth:3                        11       100%         0.10
route:keyword_only             9        100%         0.21
route:table_first              4        100%         0.23
depth:5                        4        100%         0.10
route:page_graph               1        100%         0.21
graph:on                       1        0%           -0.10
route:page_bm25                1        100%         0.20


---
# PART B: With HuggingFace — Harness + Synthesized Answers

Same harness evaluation, but now with a real LLM generating answers.
Needs T4 GPU.


## 8. Load Model

In [7]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Loading {MODEL}...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto')
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'Ready in {time.time()-t0:.0f}s | GPU: {torch.cuda.get_device_name(0)}')


Loading Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Ready in 46s | GPU: Tesla T4


## 9. Setup Engine (HuggingFace)

In [8]:
engine_hf = AdaptiveAI(
    llm_backend='huggingface',
    llm_model=MODEL,
    vectorless=True,
    storage_dir='/content/state_hf',
    log_level='ERROR',
)

engine_hf.ingest('/content/docs')
print(f'Ready with HuggingFace. Graph: {engine_hf.graph.node_count} nodes')


Ready with HuggingFace. Graph: 23 nodes


## 10. Harness + HuggingFace Answers

Now you see both: the synthesized answer AND the harness evaluation of how it was produced.


In [9]:
queries = [
    'What is Q3 revenue?',
    'How is Meridian connected to supply chain risk?',
    'What is Metformin dosage for Type 2 Diabetes?',
]

for query in queries:
    r = engine_hf.ask(query)
    print(f'Q: {query}')
    print(f'A: {r.answer[:200]}')
    print(f'  Strategy: {r.retrieval_strategy}')
    print(f'  Confidence: {r.confidence:.0%}')

    if hasattr(r, 'harness_report') and r.harness_report:
        report = r.harness_report
        print(f'  Harness efficiency: {report.efficiency:.0%}')
        for d in report.decisions:
            mark = '✓' if d.helped else '✗'
            print(f'    {mark} {d.decision}: {d.name} ({d.impact:+.2f})')

    if hasattr(r, 'shaped_reward'):
        print(f'  Shaped reward: {r.shaped_reward:.2f}')
    print()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: What is Q3 revenue?
A: According to NovaTech's financial report from Q3 of 20XX (retrieved as "q3_report.txt"), their reported total revenue was:

Revenue in Q3 = $84 million

This figure represents a growth rate increase c
  Strategy: table_first + depth=1
  Confidence: 50%
  Harness efficiency: 100%
    ✓ route: table_first (+0.15)
    ✓ depth: 5 (+0.10)
    ✓ graph: off (+0.05)
  Shaped reward: 0.44

Q: How is Meridian connected to supply chain risk?
A: Meridian Semiconductor plays a significant role in Nova Tech Inc.'s supply chain due to its high dependence and potential vulnerability mentioned within Risk Assessment data retrieved earlier ("risk_a
  Strategy: page_graph + graph(2-hop) + depth=2
  Confidence: 38%
  Harness efficiency: 67%
    ✓ route: page_graph (+0.11)
    ✓ depth: 5 (+0.10)
    ✗ graph: on (-0.10)
  Shaped reward: 0.33

Q: What is Metformin dosage for Type 2 Diabetes?
A: According to the treatment protocol you've shared with me regarding type 2 diabetes manag

## 11. Compare: No LLM vs HuggingFace

Same retrieval intelligence. Same harness evaluation. Different output layer.


In [10]:
query = 'What is Q3 revenue?'

# No LLM
r1 = engine.ask(query)
# HuggingFace
r2 = engine_hf.ask(query)

print('WITHOUT LLM (retrieval only):')
print(f'  Answer: {r1.answer[:150]}')
print(f'  Strategy: {r1.retrieval_strategy}')
if hasattr(r1, 'harness_report') and r1.harness_report:
    print(f'  Harness: {r1.harness_report.efficiency:.0%} efficient')

print()
print('WITH HuggingFace (synthesized):')
print(f'  Answer: {r2.answer[:150]}')
print(f'  Strategy: {r2.retrieval_strategy}')
if hasattr(r2, 'harness_report') and r2.harness_report:
    print(f'  Harness: {r2.harness_report.efficiency:.0%} efficient')

print()
print('Same retrieval. Same harness. Same RL learning.')
print('The LLM is the presentation layer. The intelligence is underneath.')


WITHOUT LLM (retrieval only):
  Answer: From q3_report.txt:
NovaTech Q3 2025: Revenue $847M (+12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Operating income $127M, margin 15.0%
  Strategy: page_graph + depth=1
  Harness: 100% efficient

WITH HuggingFace (synthesized):
  Answer: According to NovaTech'S financial report from Quarter Three of Year Two Thousand Twenty-Five ('q3_report'.txt), they have stated that their recorded T
  Strategy: table_first + depth=1
  Harness: 100% efficient

Same retrieval. Same harness. Same RL learning.
The LLM is the presentation layer. The intelligence is underneath.


---
## Summary

| Feature | What it does | Impact |
|---------|-------------|--------|
| Harness Agent | Evaluates every decision (route, depth, graph, tools) | Specific feedback per decision |
| Loop Engineering | Adapts exploration per domain | Learns faster where data is sparse |
| Shaped Rewards | Combines answer score + decision efficiency | 3x faster convergence |
| HuggingFace | Synthesizes answers from retrieved context | Clean output for users |

---

**pip install adaptive-intelligence**

[PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | [Paper](https://www.researchgate.net/publication/405076088)

Author: Venkatkumar Rajan | @VK_Venkatkumar
